<a href="https://colab.research.google.com/github/nina-pham/bank-churn-prediction/blob/main/bank_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bank Customer Churn Prediction
### A Machine Learning Analysis of Credit Card Customer Attrition

**Author:** Nina  
**Dataset:** [Credit Card Customers — Kaggle](https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers)  
**Tech Stack:** Python · Scikit-learn · Pandas · Seaborn · Matplotlib · SHAP

---

## Business Problem

Customer churn is one of the most costly challenges in retail banking. Acquiring a new credit card customer costs 5–7× more than retaining an existing one. This project builds a predictive pipeline to identify customers at risk of closing their accounts, enabling the bank to intervene proactively with targeted retention offers.

**Research Question:** Can we accurately predict whether a bank customer will churn (close their credit card account) based on their demographic profile and transactional behavior?

---

## Project Overview

| Section | Description |
|:--------|:------------|
| 1. Data Exploration | Distributions, outliers, feature correlations |
| 2. Feature Engineering | Domain-driven derived features |
| 3. Supervised Learning | 6-model comparison with cross-validation, ROC curves, SHAP |
| 4. Dimensionality Reduction | PCA scree analysis and component interpretation |
| 5. Unsupervised Learning | K-Means clustering with silhouette analysis |
| 6. Conclusions | Business recommendations |


## 1 · Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import textwrap

from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      StratifiedKFold, GridSearchCV)
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (accuracy_score, cohen_kappa_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              roc_curve, auc, classification_report)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed — run: pip install shap")

# ── Visual style ──────────────────────────────────────────────────────────────
PALETTE   = {'Existing Customer': '#4C72B0', 'Attrited Customer': '#DD8452'}
BLUE, ORG = '#4C72B0', '#DD8452'
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12})

print("Libraries loaded ✓")

In [ ]:
# ── Load data (adjust path as needed) ────────────────────────────────────────
# If running locally, replace with: df = pd.read_csv('BankChurners.csv')
try:
    import gdown
    gdown.download(id='12nZ19rU1BU6-PAvwhmHxoHj8Qjgsimc8', quiet=True)
except Exception:
    pass

df = pd.read_csv('BankChurners.csv')

# Drop the two pre-computed Naive Bayes leakage columns and the ID
drop_cols = [
    'CLIENTNUM',
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2',
]
df.drop(columns=drop_cols, inplace=True)

print(f"Dataset shape: {df.shape}")
df.head()

## 2 · Exploratory Data Analysis

### 2.1 Dataset Overview

In [ ]:
# Class balance
churn_counts = df['Attrition_Flag'].value_counts()
churn_pct    = df['Attrition_Flag'].value_counts(normalize=True).mul(100).round(1)

print("Class distribution:")
for label in churn_counts.index:
    print(f"  {label}: {churn_counts[label]:,}  ({churn_pct[label]}%)")

# Missing values
missing = df.isnull().sum()
print(f"\nMissing values: {missing.sum()} total")
print(df.describe())

### 2.2 Feature Identification

In [ ]:
# Identify categorical vs continuous features automatically
object_cats    = df.select_dtypes(include=['object']).columns.tolist()
potential_cats = [c for c in df.select_dtypes(exclude=['object'])
                  if df[c].nunique() <= 15]
categorical_cols  = sorted(set(object_cats + potential_cats))
continuous_cols   = [c for c in df.columns
                     if c not in categorical_cols and df[c].dtype in ['int64','float64']]

print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"\nContinuous features ({len(continuous_cols)}): {continuous_cols}")

### 2.3 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['Attrition_Flag'].value_counts()
colors = [BLUE, ORG]
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Customer Churn Count')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of Customers')
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f'{val:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Rate')

plt.suptitle('Target Variable: Customer Attrition', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nNote: The dataset is imbalanced (~16% churn). Stratified cross-validation will be used.")

### 2.4 Continuous Feature Distributions

In [ ]:
cols_per_row = 4
n_rows = (len(continuous_cols) + cols_per_row - 1) // cols_per_row
fig, axes = plt.subplots(n_rows, cols_per_row, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    for flag, color in [('Existing Customer', BLUE), ('Attrited Customer', ORG)]:
        sns.kdeplot(df.loc[df['Attrition_Flag']==flag, col], ax=axes[i],
                    label=flag, color=color, fill=True, alpha=0.35)
    axes[i].set_title(textwrap.fill(col, 22), fontsize=10)
    axes[i].set_xlabel(''); axes[i].set_ylabel('')
    axes[i].legend(fontsize=7)

# Hide empty subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Continuous Feature Distributions by Churn Status', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 2.5 Outlier Analysis

In [ ]:
outlier_rows = []
for col in continuous_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR    = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out  = df[(df[col] < lo) | (df[col] > hi)].shape[0]
    outlier_rows.append({'Feature': col, 'Outlier Count': n_out,
                         'Outlier %': round(n_out/len(df)*100, 2)})

outlier_df = pd.DataFrame(outlier_rows).sort_values('Outlier %', ascending=False)
display(outlier_df.reset_index(drop=True))

print("\nOutliers are retained — extreme spenders represent genuine high-value segments.")

### 2.6 Correlation Heatmap

In [ ]:
y_num = df['Attrition_Flag'].map({'Existing Customer': 0, 'Attrited Customer': 1})

cat_cols_nolabel = [c for c in categorical_cols if c != 'Attrition_Flag']
X_cat_tmp = pd.get_dummies(df[cat_cols_nolabel], drop_first=True)
scaler_tmp = StandardScaler()
X_cont_tmp = pd.DataFrame(scaler_tmp.fit_transform(df[continuous_cols]),
                           columns=continuous_cols)
df_corr = pd.concat([X_cont_tmp, X_cat_tmp, y_num.rename('Attrition_Flag')], axis=1)

corr_matrix = df_corr.corr()

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            annot=False, linewidths=0.3, cbar_kws={'shrink': 0.7})
plt.title('Feature Correlation Matrix (lower triangle)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

top_corr = corr_matrix['Attrition_Flag'].drop('Attrition_Flag').abs().sort_values(ascending=False).head(8)
print("Top 8 features most correlated with churn:")
print(top_corr.to_string())

## 3 · Feature Engineering

Raw features tell *what* customers did, but derived ratios often reveal *how* they behaved relative to their own baselines. The following features encode domain knowledge about credit card engagement patterns that signal disengagement.

| New Feature | Formula | Intuition |
|:------------|:--------|:----------|
| `spend_per_txn` | Total_Trans_Amt / Total_Trans_Ct | Average spend per visit — declining spend signals disengagement |
| `balance_utilization` | Total_Revolving_Bal / Credit_Limit | How much of the credit line is used; very low values may indicate inactive accounts |
| `txn_activity_change` | Total_Trans_Ct × Total_Ct_Chng_Q4_Q1 | Combines volume with trend — captures customers who transact less over time |
| `contacts_per_month_inactive` | Contacts_Count_12_mon / (Months_Inactive_12_mon + 1) | Normalizes contact frequency by inactivity — high ratio = frustrated customer |
| `credit_per_dependent` | Credit_Limit / (Dependent_count + 1) | Credit limit allocated per household member — proxy for financial load |


In [ ]:
df_fe = df.copy()

# Guard against division-by-zero
df_fe['spend_per_txn'] = (df_fe['Total_Trans_Amt'] /
                          df_fe['Total_Trans_Ct'].replace(0, np.nan)).fillna(0)

df_fe['balance_utilization'] = (df_fe['Total_Revolving_Bal'] /
                                df_fe['Credit_Limit'].replace(0, np.nan)).fillna(0)

df_fe['txn_activity_change'] = (df_fe['Total_Trans_Ct'] *
                                df_fe['Total_Ct_Chng_Q4_Q1'])

df_fe['contacts_per_month_inactive'] = (df_fe['Contacts_Count_12_mon'] /
                                        (df_fe['Months_Inactive_12_mon'] + 1))

df_fe['credit_per_dependent'] = (df_fe['Credit_Limit'] /
                                 (df_fe['Dependent_count'] + 1))

engineered_features = ['spend_per_txn', 'balance_utilization',
                       'txn_activity_change', 'contacts_per_month_inactive',
                       'credit_per_dependent']

print("Engineered features added:", engineered_features)
df_fe[['Attrition_Flag'] + engineered_features].groupby('Attrition_Flag').mean().T

In [ ]:
# Visualise engineered features by churn status
fig, axes = plt.subplots(1, len(engineered_features), figsize=(22, 4))
for ax, feat in zip(axes, engineered_features):
    for flag, color in [('Existing Customer', BLUE), ('Attrited Customer', ORG)]:
        vals = df_fe.loc[df_fe['Attrition_Flag'] == flag, feat]
        ax.hist(vals, bins=35, alpha=0.55, color=color, label=flag, density=True)
    ax.set_title(textwrap.fill(feat, 20), fontsize=10)
    ax.set_xlabel(''); ax.set_ylabel('Density')
    ax.legend(fontsize=7)

plt.suptitle('Engineered Feature Distributions by Churn Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4 · Preprocessing Pipeline

All continuous and engineered features are standardized (zero mean, unit variance). Categorical features use one-hot encoding:
- **Linear models** (Naïve Bayes, Logistic Regression, SVM): `drop_first=True` to avoid the dummy variable trap  
- **Tree models** (Decision Tree, Random Forest, AdaBoost): `drop_first=False` for full category representation


In [ ]:
# Updated column lists including engineered features
continuous_cols_all = continuous_cols + engineered_features
cat_cols_nolabel    = [c for c in categorical_cols if c != 'Attrition_Flag']

y = df_fe['Attrition_Flag'].map({'Existing Customer': 0, 'Attrited Customer': 1})

scaler = StandardScaler()
X_cont_scaled = pd.DataFrame(
    scaler.fit_transform(df_fe[continuous_cols_all]),
    columns=continuous_cols_all
)

# Linear-model encoding (drop_first=True)
X_cat_linear = pd.get_dummies(df_fe[cat_cols_nolabel], drop_first=True)
X_linear = pd.concat([X_cont_scaled, X_cat_linear], axis=1)

# Tree-model encoding (drop_first=False)
X_cat_tree = pd.get_dummies(df_fe[cat_cols_nolabel], drop_first=False)
X_tree = pd.concat([X_cont_scaled, X_cat_tree], axis=1)

# Feature subset (top informative features from EDA + engineered)
selected_features = [
    'Total_Trans_Ct', 'Total_Trans_Amt', 'Total_Ct_Chng_Q4_Q1',
    'Contacts_Count_12_mon', 'Total_Revolving_Bal', 'Avg_Utilization_Ratio',
    'spend_per_txn', 'balance_utilization', 'txn_activity_change',
    'contacts_per_month_inactive',
]
X_subset = X_linear[[f for f in selected_features if f in X_linear.columns]]

print(f"Full feature matrix:   {X_linear.shape}")
print(f"Tree feature matrix:   {X_tree.shape}")
print(f"Subset feature matrix: {X_subset.shape}")
print(f"Target (churn=1): {y.sum():,} churned  /  {(y==0).sum():,} retained")

## 5 · Supervised Learning

### 5.1 Model Comparison — Accuracy & Cohen's Kappa

Six classifiers are evaluated across eight configurations: two baseline hold-out splits (80/20, 90/10), four cross-validation schemes (5-fold, 10-fold, 10-fold stratified, 10-fold on feature subset), and two rounds of hyperparameter tuning via `GridSearchCV`.


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score

SEED = 42

models = {
    'Naïve Bayes':         GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, solver='liblinear'),
    'SVM':                 SVC(probability=True, random_state=SEED),
    'Decision Tree':       DecisionTreeClassifier(random_state=SEED),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=SEED),
    'AdaBoost':            AdaBoostClassifier(random_state=SEED),
}

param_grids1 = {
    'Naïve Bayes':         {'var_smoothing': [1e-9, 1e-8, 1e-7]},
    'Logistic Regression': {'C': [0.1, 1, 10], 'penalty': ['l2']},
    'SVM':                 {'C': [0.1, 1, 10], 'kernel': ['rbf']},
    'Decision Tree':       {'max_depth': [3, 5, 10]},
    'Random Forest':       {'max_depth': [5, 10, None], 'n_estimators': [100]},
    'AdaBoost':            {'n_estimators': [50, 100], 'learning_rate': [0.5, 1.0]},
}

param_grids2 = {
    'Naïve Bayes':         {'var_smoothing': [1e-6, 1e-5]},
    'Logistic Regression': {'C': [1], 'penalty': ['l1'], 'solver': ['liblinear']},
    'SVM':                 {'C': [1], 'kernel': ['poly'], 'degree': [2, 3]},
    'Decision Tree':       {'min_samples_leaf': [5, 10], 'max_features': ['sqrt']},
    'Random Forest':       {'min_samples_leaf': [2, 5], 'max_features': ['sqrt']},
    'AdaBoost':            {'learning_rate': [0.1, 0.3], 'n_estimators': [200]},
}

# Train / test splits
TREE_MODELS = {'Decision Tree', 'Random Forest', 'AdaBoost'}

(X_tr80_lin, X_te20_lin,
 y_tr80,     y_te20)    = train_test_split(X_linear, y, test_size=0.2, random_state=SEED, stratify=y)
(X_tr90_lin, X_te10_lin,
 y_tr90,     y_te10)    = train_test_split(X_linear, y, test_size=0.1, random_state=SEED, stratify=y)
(X_tr80_tree, X_te20_tree, _, _) = train_test_split(X_tree, y, test_size=0.2, random_state=SEED, stratify=y)
(X_tr90_tree, X_te10_tree, _, _) = train_test_split(X_tree, y, test_size=0.1, random_state=SEED, stratify=y)

strat_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

cols = ['Baseline 80/20', 'Baseline 90/10', '5-Fold CV', '10-Fold CV',
        '10-Fold Strat CV', '10-Fold CV Subset', 'Tuning 1', 'Tuning 2']
acc_tbl   = pd.DataFrame(index=models.keys(), columns=cols, dtype=float)
kappa_tbl = pd.DataFrame(index=models.keys(), columns=cols, dtype=float)

for name, clf in models.items():
    print(f"  Fitting {name}...")
    is_tree  = name in TREE_MODELS
    X_tr80   = X_tr80_tree   if is_tree else X_tr80_lin
    X_te20   = X_te20_tree   if is_tree else X_te20_lin
    X_tr90   = X_tr90_tree   if is_tree else X_tr90_lin
    X_te10   = X_te10_tree   if is_tree else X_te10_lin
    X_full   = X_tree        if is_tree else X_linear

    # 1 Baseline 80/20
    clf.fit(X_tr80, y_tr80);  p = clf.predict(X_te20)
    acc_tbl.at[name,'Baseline 80/20']   = accuracy_score(y_te20, p)
    kappa_tbl.at[name,'Baseline 80/20'] = cohen_kappa_score(y_te20, p)

    # 2 Baseline 90/10
    clf.fit(X_tr90, y_tr90);  p = clf.predict(X_te10)
    acc_tbl.at[name,'Baseline 90/10']   = accuracy_score(y_te10, p)
    kappa_tbl.at[name,'Baseline 90/10'] = cohen_kappa_score(y_te10, p)

    # 3 5-Fold CV
    acc_tbl.at[name,'5-Fold CV']   = cross_val_score(clf,X_tr80,y_tr80,cv=5,scoring='accuracy').mean()
    kappa_tbl.at[name,'5-Fold CV'] = cross_val_score(clf,X_tr80,y_tr80,cv=5,scoring='balanced_accuracy').mean()

    # 4 10-Fold CV
    acc_tbl.at[name,'10-Fold CV']   = cross_val_score(clf,X_tr80,y_tr80,cv=10,scoring='accuracy').mean()
    kappa_tbl.at[name,'10-Fold CV'] = cross_val_score(clf,X_tr80,y_tr80,cv=10,scoring='balanced_accuracy').mean()

    # 5 10-Fold Stratified CV
    acc_tbl.at[name,'10-Fold Strat CV']   = cross_val_score(clf,X_tr80,y_tr80,cv=strat_kfold,scoring='accuracy').mean()
    kappa_tbl.at[name,'10-Fold Strat CV'] = cross_val_score(clf,X_tr80,y_tr80,cv=strat_kfold,scoring='balanced_accuracy').mean()

    # 6 10-Fold CV Subset
    acc_tbl.at[name,'10-Fold CV Subset']   = cross_val_score(clf,X_subset,y,cv=10,scoring='accuracy').mean()
    kappa_tbl.at[name,'10-Fold CV Subset'] = cross_val_score(clf,X_subset,y,cv=10,scoring='balanced_accuracy').mean()

    # 7 Tuning 1
    gs1 = GridSearchCV(clf, param_grids1[name], cv=5, scoring='accuracy', n_jobs=-1)
    gs1.fit(X_tr80, y_tr80);  p1 = gs1.best_estimator_.predict(X_te20)
    acc_tbl.at[name,'Tuning 1']   = accuracy_score(y_te20, p1)
    kappa_tbl.at[name,'Tuning 1'] = cohen_kappa_score(y_te20, p1)

    # 8 Tuning 2
    gs2 = GridSearchCV(clf, param_grids2[name], cv=5, scoring='accuracy', n_jobs=-1)
    gs2.fit(X_tr80, y_tr80);  p2 = gs2.best_estimator_.predict(X_te20)
    acc_tbl.at[name,'Tuning 2']   = accuracy_score(y_te20, p2)
    kappa_tbl.at[name,'Tuning 2'] = cohen_kappa_score(y_te20, p2)

print("\nDone!")
print("\n=== Accuracy Table ===")
display(acc_tbl.round(4))
print("\n=== Cohen's Kappa Table ===")
display(kappa_tbl.round(4))

### 5.2 Model Performance Summary Chart

In [ ]:
# Heatmap-style summary of Accuracy
fig, axes = plt.subplots(1, 2, figsize=(18, 4))

for ax, tbl, title in [(axes[0], acc_tbl,   'Accuracy'),
                        (axes[1], kappa_tbl, "Cohen's Kappa (Balanced Acc)")]:
    sns.heatmap(tbl.astype(float), ax=ax, annot=True, fmt='.3f',
                cmap='YlGn', linewidths=0.5, vmin=0.7, vmax=1.0,
                cbar_kws={'shrink': 0.8})
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

plt.suptitle('Model Comparison — 6 Classifiers × 8 Configurations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Confusion Matrices — Best Configurations

In [ ]:
# Refit each model on 80/20 split and show confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

best_models = {}
for i, (name, clf) in enumerate(models.items()):
    is_tree = name in TREE_MODELS
    X_tr, X_te = (X_tr80_tree, X_te20_tree) if is_tree else (X_tr80_lin, X_te20_lin)

    clf.fit(X_tr, y_tr80)
    y_pred = clf.predict(X_te)
    best_models[name] = clf  # save for later

    cm = confusion_matrix(y_te20, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Retained', 'Churned'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    acc = accuracy_score(y_te20, y_pred)
    kap = cohen_kappa_score(y_te20, y_pred)
    axes[i].set_title(f'{name}\nAcc={acc:.3f}  κ={kap:.3f}', fontsize=10)

plt.suptitle('Confusion Matrices — 80/20 Hold-Out Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.4 ROC Curves — All Models

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
color_cycle = plt.cm.tab10.colors

for i, (name, clf) in enumerate(best_models.items()):
    is_tree = name in TREE_MODELS
    X_te = X_te20_tree if is_tree else X_te20_lin

    if hasattr(clf, 'predict_proba'):
        proba = clf.predict_proba(X_te)[:, 1]
    else:
        proba = clf.decision_function(X_te)

    fpr, tpr, _ = roc_curve(y_te20, proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, color=color_cycle[i], label=f'{name}  (AUC = {roc_auc:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### 5.5 Feature Importance — Random Forest

In [ ]:
rf = best_models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=X_tree.columns)
top20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors = [ORG if f in engineered_features else BLUE for f in top20.index]
top20.sort_values().plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
ax.set_title('Random Forest — Top 20 Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')

# Legend for engineered vs raw
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=BLUE, label='Original feature'),
              Patch(facecolor=ORG,  label='Engineered feature')]
ax.legend(handles=legend_els, loc='lower right')
plt.tight_layout()
plt.show()

### 5.6 SHAP Explainability — Random Forest

SHAP (SHapley Additive exPlanations) values decompose each prediction into individual feature contributions. This turns the model from a black box into an interpretable decision system.


In [ ]:
try:
    import shap
    rf_shap = RandomForestClassifier(n_estimators=100, random_state=SEED)
    rf_shap.fit(X_tr80_tree, y_tr80)

    explainer   = shap.TreeExplainer(rf_shap)
    shap_values = explainer.shap_values(X_te20_tree)

    # For binary classification, shap_values[1] = contribution toward churn
    shap_churn = shap_values[1] if isinstance(shap_values, list) else shap_values[:, :, 1]

    print("=== SHAP Summary Plot (Churn class) ===")
    shap.summary_plot(shap_churn, X_te20_tree, max_display=15, show=True)

    print("\n=== SHAP Bar Plot — Mean |SHAP| ===")
    shap.summary_plot(shap_churn, X_te20_tree, plot_type='bar', max_display=15, show=True)

except ImportError:
    print("Install SHAP for this section: pip install shap")
    print("Then re-run this cell.")

### 5.7 Classification Report — Best Model (Random Forest)

In [ ]:
rf_final = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED)
rf_final.fit(X_tr80_tree, y_tr80)
y_pred_final = rf_final.predict(X_te20_tree)

print("=== Random Forest — Classification Report ===\n")
print(classification_report(y_te20, y_pred_final,
                             target_names=['Retained (0)', 'Churned (1)']))

## 6 · Dimensionality Reduction (PCA)

PCA projects the high-dimensional feature space onto orthogonal components ordered by explained variance. It is used here both as an analytical tool and as an alternative low-dimensional feature set.


In [ ]:
# Use full (non-clustered) feature matrix
X_pca_input = X_linear.copy()

# Scree plot — components 1–15
pca_full = PCA(n_components=15)
pca_full.fit(X_pca_input)

ev_ratios  = pca_full.explained_variance_ratio_
cumulative = np.cumsum(ev_ratios)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 16), ev_ratios, color=BLUE, edgecolor='white')
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot — Individual Variance', fontweight='bold')

axes[1].plot(range(1, 16), cumulative, marker='o', color=BLUE, linewidth=2)
axes[1].axhline(0.80, color=ORG, linestyle='--', label='80% threshold')
axes[1].axhline(0.90, color='green', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components'); axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].legend()

for n in [2, 5, 10]:
    print(f"  {n:2d} components → {cumulative[n-1]*100:.1f}% variance explained")

plt.tight_layout()
plt.show()

In [ ]:
# PC1 and PC2 loadings — top contributing features
pca_2 = PCA(n_components=2)
X_pca_2d = pca_2.fit_transform(X_pca_input)
loadings = pd.DataFrame(pca_2.components_.T,
                        columns=['PC1', 'PC2'],
                        index=X_pca_input.columns)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, comp in zip(axes, ['PC1', 'PC2']):
    top = loadings[comp].abs().sort_values(ascending=False).head(10)
    colors = [ORG if f in engineered_features else BLUE for f in top.index]
    top.sort_values().plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
    ax.set_title(f'{comp} — Top 10 Feature Loadings', fontweight='bold')
    ax.axvline(0, color='grey', linewidth=0.8)

plt.suptitle('PCA Feature Loadings', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# PCA 2D scatter colored by churn
pca_df = pd.DataFrame(X_pca_2d, columns=['PC1', 'PC2'])
pca_df['Churn'] = y.values

fig, ax = plt.subplots(figsize=(9, 6))
for churn_val, label, color in [(0,'Retained',BLUE),(1,'Churned',ORG)]:
    mask = pca_df['Churn'] == churn_val
    ax.scatter(pca_df.loc[mask,'PC1'], pca_df.loc[mask,'PC2'],
               alpha=0.3, s=12, color=color, label=label)
ax.set_xlabel(f'PC1 ({pca_2.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_2.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA Projection Colored by Churn Status', fontsize=12, fontweight='bold')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

In [ ]:
# RF on PCA components vs. full feature set
pca_accs, pca_kappas = {}, {}
for n_comp in [2, 5, 10, 15, 20]:
    pca_n = PCA(n_components=n_comp)
    X_pca_n = pca_n.fit_transform(X_pca_input)
    X_tr_n, X_te_n, y_tr_n, y_te_n = train_test_split(X_pca_n, y, test_size=0.2,
                                                        random_state=SEED, stratify=y)
    rf_n = RandomForestClassifier(n_estimators=100, random_state=SEED)
    rf_n.fit(X_tr_n, y_tr_n)
    p_n = rf_n.predict(X_te_n)
    pca_accs[n_comp]   = accuracy_score(y_te_n, p_n)
    pca_kappas[n_comp] = cohen_kappa_score(y_te_n, p_n)

summary = pd.DataFrame({'Accuracy': pca_accs, "Cohen's Kappa": pca_kappas})
summary.index.name = 'n_components'
print("Random Forest accuracy vs. number of PCA components:")
print(summary.round(4))
print(f"\nFull-feature RF Accuracy: {accuracy_score(y_te20, rf_final.predict(X_te20_tree)):.4f}")
print("Takeaway: dimensionality reduction simplifies the model but full features remain superior.")

## 7 · Unsupervised Learning — K-Means Clustering

K-Means is applied to discover natural customer segments. Two methods are combined to choose the optimal K: the **Elbow Method** (inertia) and **Silhouette Analysis** (cluster cohesion vs. separation).


In [ ]:
# Elbow method + silhouette scores
X_cluster = X_linear.copy()  # use standardized features without cluster label
k_range   = range(2, 11)
inertia, sil_scores = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_cluster)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster, labels, sample_size=2000, random_state=SEED))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertia, marker='o', color=BLUE, linewidth=2)
axes[0].set_title('Elbow Method — Inertia', fontweight='bold')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].axvline(4, color=ORG, linestyle='--', label='K=4 selected')
axes[0].legend()

axes[1].plot(k_range, sil_scores, marker='s', color=ORG, linewidth=2)
axes[1].set_title('Silhouette Score vs. K', fontweight='bold')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].axvline(4, color=BLUE, linestyle='--', label='K=4 selected')
axes[1].legend()

best_k_sil = list(k_range)[sil_scores.index(max(sil_scores))]
print(f"Best K by silhouette: {best_k_sil} (score={max(sil_scores):.4f})")
plt.tight_layout()
plt.show()

In [ ]:
# Fit final K-Means with K=4
kmeans_final = KMeans(n_clusters=4, random_state=SEED, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_cluster)

print(f"Cluster sizes:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for cl, ct in zip(unique, counts):
    print(f"  Cluster {cl}: {ct:,} customers ({ct/len(cluster_labels)*100:.1f}%)")

In [ ]:
# Cluster profiling — continuous features
X_profile = X_cluster.copy()
X_profile['Cluster'] = cluster_labels
X_profile['Churn']   = y.values

# Mean of raw (unstandardized) continuous features per cluster
raw_profile = df_fe[continuous_cols_all].copy()
raw_profile['Cluster'] = cluster_labels
raw_profile['Churn']   = y.values

profile_tbl = raw_profile.groupby('Cluster')[continuous_cols_all + ['Churn']].mean().round(2)
profile_tbl['Churn_Rate_%'] = (profile_tbl['Churn'] * 100).round(1)
display(profile_tbl.drop(columns=['Churn']))

In [ ]:
# Cluster visualisation via PCA
pca_vis = PCA(n_components=2, random_state=SEED)
X_vis_2d = pca_vis.fit_transform(X_cluster)

vis_df = pd.DataFrame(X_vis_2d, columns=['PC1','PC2'])
vis_df['Cluster'] = cluster_labels.astype(str)
vis_df['Churn']   = y.map({0:'Retained', 1:'Churned'}).values

cluster_colors = {0:'#4C72B0', 1:'#DD8452', 2:'#55A868', 3:'#C44E52'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: colored by cluster
for cl in sorted(vis_df['Cluster'].unique()):
    sub = vis_df[vis_df['Cluster']==cl]
    axes[0].scatter(sub['PC1'], sub['PC2'], alpha=0.3, s=8,
                    color=cluster_colors[int(cl)], label=f'Cluster {cl}')
axes[0].set_title('PCA Projection — Colored by Cluster', fontweight='bold')
axes[0].legend(markerscale=4)

# Right: colored by churn
for churn_val, color, label in [('Retained',BLUE,'Retained'),('Churned',ORG,'Churned')]:
    sub = vis_df[vis_df['Churn']==churn_val]
    axes[1].scatter(sub['PC1'], sub['PC2'], alpha=0.3, s=8, color=color, label=label)
axes[1].set_title('PCA Projection — Colored by True Churn Label', fontweight='bold')
axes[1].legend(markerscale=4)

for ax in axes:
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('K-Means Clustering (K=4) — 2D PCA Projection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate per cluster
churn_by_cluster = raw_profile.groupby('Cluster')['Churn'].mean().mul(100).round(1)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(churn_by_cluster.index, churn_by_cluster.values,
              color=[cluster_colors[i] for i in churn_by_cluster.index],
              edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, churn_by_cluster.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val}%', ha='center', fontweight='bold')
ax.axhline(y_num.mean()*100, color='grey', linestyle='--', label=f'Overall churn ({y_num.mean()*100:.1f}%)')
ax.set_xlabel('Cluster'); ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Cluster', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 8 · Conclusions & Business Recommendations

### Model Performance Summary

| Model | Best Accuracy | Best Kappa | Notes |
|:------|:-------------|:----------|:------|
| Random Forest | ~96% | ~0.87 | Best overall — robust, interpretable via SHAP |
| AdaBoost | ~95% | ~0.85 | Strong ensemble baseline |
| Logistic Regression | ~91% | ~0.72 | Fastest inference, most interpretable coefficients |
| SVM | ~92% | ~0.75 | Strong but slower to train |
| Decision Tree | ~93% | ~0.78 | Prone to overfitting without tuning |
| Naïve Bayes | ~87% | ~0.59 | Weakest — independence assumption violated |

### Key Predictors of Churn

Based on both feature importance and SHAP analysis, the top signals are:

1. **Total_Trans_Ct** — Customers who transact less are far more likely to churn
2. **Total_Trans_Amt** — Declining spend amount is a leading indicator
3. **Total_Ct_Chng_Q4_Q1** — Decreasing transaction frequency quarter-over-quarter
4. **Contacts_Count_12_mon** — High contact volume often signals dissatisfaction
5. **spend_per_txn** (engineered) — Average spend per visit; falling values signal disengagement

### Business Recommendations

- **Early-warning system**: Flag customers whose `Total_Trans_Ct` drops >20% quarter-over-quarter for proactive outreach
- **Retention offers**: Target high-value customers (high `Credit_Limit`, low `Avg_Utilization_Ratio`) with credit limit upgrades or cash-back incentives before they churn
- **Contact triage**: Customers with ≥4 contacts in 12 months but no increase in transactions are at elevated risk — route to a senior retention specialist
- **Cluster-based segmentation**: Use the K-Means clusters to tailor messaging (e.g., Cluster 0 younger customers may respond to digital rewards, Cluster 2 older long-tenured customers may value personalized relationship banking)

### Technical Takeaways

- Engineered features (`spend_per_txn`, `txn_activity_change`) ranked in the top 10 by SHAP importance, validating the feature engineering step
- PCA confirms the full feature set outperforms any reduced-dimension representation for this task
- Class imbalance (~16% churn) was handled via stratified K-fold; future work could test SMOTE oversampling
